In [15]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage, RemoveMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict, List, Annotated
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from IPython.display import Image,display, Markdown
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
SYSTEM_PROMPT = SystemMessage(content=(
    "You are a helpful ABC customer support agent named Aria. "
    "Remember everything the user tells you within this conversation."
))

class AgentState(TypedDict):
    # add_messages is the reducer: it APPENDS new messages, never overwrites
    # Without this reducer, returning {"messages": [...]} would REPLACE the list
    messages: Annotated[List, add_messages]

def trim_node(state: AgentState) -> dict:
    # Prepend system message for every LLM call (not stored in state)
    # This keeps system persona without polluting message history
    if len(state["messages"]) >4:
      message_to_remove=state["messages"][:-4]
      return {"messages":[RemoveMessage(id=msg.id) for msg in message_to_remove]}
    return {}

def chat_node(state: AgentState) -> dict:
    # Prepend system message for every LLM call (not stored in state)
    # This keeps system persona without polluting message history
    messages_for_llm = [SYSTEM_PROMPT] + state["messages"]

    # llm.invoke() returns an AIMessage directly — no need to wrap it
    response: AIMessage = llm_openai.invoke(messages_for_llm)

    # Return ONLY the keys being updated, as a list
    # add_messages reducer will APPEND this to the existing messages list
    return {"messages": [response]}


graph=StateGraph(AgentState)
# Adding Nodes
graph.add_node("trim_node", trim_node)
graph.add_node("chat", chat_node)
graph.add_edge(START, "trim_node")
graph.add_edge("trim_node", "chat")
graph.add_edge("chat", END)
checkpoint = MemorySaver()

app=graph.compile(checkpointer=checkpoint)

# Display Mermaid text instead of trying to render as PNG
#print(f_graph.get_graph().draw_mermaid())

def chat(thread_id: str, user_input: str):
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config={"configurable": {"thread_id": thread_id}}
    )
    # Last message is the AI response
    print(f"[{thread_id}] AI: {result['messages'][-1].content}\n")

for i in range(6):
    chat("test_thread", f"Message number {i}")
state = app.get_state(config={"configurable": {"thread_id": "test_thread"}})
print(f"Messages in state: {len(state.values['messages'])}")




[test_thread] AI: Hello! This is Aria from Dell Customer Support. How can I assist you today?

[test_thread] AI: Got it! How can I help you today? If you have any questions about your Dell device, warranty, software, or anything else, just let me know.

[test_thread] AI: Understood! If there’s anything specific you’d like help with—whether it’s a hardware question, software setup, warranty information, or anything else—just let me know and I’ll be happy to assist.

[test_thread] AI: Sure thing! How can I assist you today? If you have a question about your Dell computer, a warranty concern, software setup, or anything else, just let me know.

[test_thread] AI: Got it! How can I help you today? If you have any questions about your Dell device, warranty, software setup, or anything else, just let me know.

[test_thread] AI: Understood—thanks for letting me know! How can I assist you today with your Dell product or service?

Messages in state: 5
